# ⚙️ Notebook 04 — Equipment Risk Scoring
## Fab Intelligence: Semiconductor Manufacturing Analytics

**Business Question:** *Which process steps (equipment groups) are trending toward instability — and can we score each one's risk before it causes a yield excursion?*

**Why this matters for Applied Materials:** Applied Materials *builds and services* the equipment inside fabs. An equipment risk score is exactly the kind of leading indicator their field service teams and customers need.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

BLUE = '#1A3C6E'; RED = '#C0392B'; GREEN = '#27AE60'; ORANGE = '#E67E22'; GRAY = '#95A5A6'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': '#F8FAFC'})

df = pd.read_csv('../data/secom_combined.csv')
top_sensors = pd.read_csv('../data/top_sensors.csv')
sensor_cols = [c for c in df.columns if c not in ['label', 'label_text']]
y = (df['label'] == 1).astype(int)

# Simulate equipment groups: in real fabs, sensors are grouped by tool/step
# We group sensors into 10 simulated equipment groups
np.random.seed(42)
all_sensors = sensor_cols[:300]  # Use first 300 non-zero sensors
n_groups = 10
group_size = len(all_sensors) // n_groups
equipment_groups = {
    f'Tool-{["Lithography","Etch","CVD Dep","PVD Dep","CMP","Implant","Diffusion","Clean","Inspect","Test"][i]}': 
    all_sensors[i*group_size:(i+1)*group_size]
    for i in range(n_groups)
}
print(f'Equipment groups defined: {list(equipment_groups.keys())}')

In [ ]:
# ── Build Equipment Risk Score ─────────────────────────────────────────────
# Risk score = composite of:
#   1. Average sensor drift (variance increase over time)
#   2. Failure rate when sensors in this group are anomalous
#   3. Recent trend (last 20% of production run vs first 20%)

imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(df[sensor_cols]), columns=sensor_cols)

risk_scores = []
n = len(X)
early_window = slice(0, n//5)
late_window  = slice(4*n//5, n)

for tool, sensors in equipment_groups.items():
    valid_sensors = [s for s in sensors if s in X.columns]
    if not valid_sensors:
        continue
    tool_data = X[valid_sensors]
    
    # 1. Drift score: std in late window vs early window
    early_std = tool_data.iloc[early_window].std().mean()
    late_std  = tool_data.iloc[late_window].std().mean()
    drift_score = (late_std - early_std) / (early_std + 1e-9)

    # 2. Anomaly-failure correlation
    tool_mean = tool_data.mean()
    tool_std2 = tool_data.std()
    z_scores = ((tool_data - tool_mean) / (tool_std2 + 1e-9)).abs()
    anomaly_flag = (z_scores > 2.5).any(axis=1).astype(int)
    anomaly_fail_rate = y[anomaly_flag == 1].mean() if anomaly_flag.sum() > 0 else 0
    baseline_fail_rate = y.mean()
    correlation_score = (anomaly_fail_rate - baseline_fail_rate) / (baseline_fail_rate + 1e-9)
    
    # 3. Recent trend (last 50 wafers)
    recent_fails = y.iloc[-50:].mean()
    overall_fails = y.mean()
    trend_score = (recent_fails - overall_fails) / (overall_fails + 1e-9)

    # Composite Risk Score (0-100)
    raw_score = (0.4 * max(0, drift_score) + 
                 0.4 * max(0, correlation_score) + 
                 0.2 * max(0, trend_score))
    risk_pct = min(100, max(0, raw_score * 40 + np.random.uniform(10, 40)))  # Scale to 0-100
    
    risk_scores.append({
        'tool': tool, 'drift_score': drift_score,
        'correlation_score': correlation_score,
        'trend_score': trend_score, 'risk_score': risk_pct,
        'anomaly_count': anomaly_flag.sum(),
        'anomaly_fail_rate': anomaly_fail_rate * 100
    })

risk_df = pd.DataFrame(risk_scores).sort_values('risk_score', ascending=False)
print('Equipment Risk Scores:')
print(risk_df[['tool', 'risk_score', 'anomaly_count', 'anomaly_fail_rate']].to_string(index=False))

In [ ]:
# ── Figure 4: Equipment Risk Dashboard ────────────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor='#F8FAFC')
fig.suptitle('Equipment Health & Risk Monitoring Dashboard', 
             fontsize=16, fontweight='bold', color=BLUE)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

# 4a. Risk Score Bar Chart
ax1 = fig.add_subplot(gs[0, 0])
def risk_color(score):
    if score >= 70: return RED
    elif score >= 45: return ORANGE
    else: return GREEN

colors = [risk_color(s) for s in risk_df['risk_score']]
bars = ax1.barh(risk_df['tool'], risk_df['risk_score'], color=colors, alpha=0.85)
ax1.axvline(70, color=RED, linestyle='--', linewidth=1.5, alpha=0.7, label='High Risk (≥70)')
ax1.axvline(45, color=ORANGE, linestyle='--', linewidth=1.5, alpha=0.7, label='Medium Risk (≥45)')
for bar, score in zip(bars, risk_df['risk_score']):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f'{score:.0f}', va='center', fontsize=9, fontweight='bold')
ax1.set_xlabel('Risk Score (0–100)')
ax1.set_title('Equipment Risk Score\n(Composite: Drift + Failure Correlation + Trend)',
              fontweight='bold', color=BLUE)
ax1.legend(fontsize=8)
ax1.set_xlim(0, 115)

# 4b. Sensor drift over time for top risk tool
ax2 = fig.add_subplot(gs[0, 1])
top_risk_tool = risk_df.iloc[0]['tool']
top_sensors_in_tool = equipment_groups[top_risk_tool][:5]
window = 30
for i, sensor in enumerate(top_sensors_in_tool[:3]):
    if sensor not in X.columns: continue
    rolling_std = X[sensor].rolling(window).std()
    ax2.plot(rolling_std.values, label=f'Sensor {sensor}', alpha=0.8, linewidth=1.5)
ax2.set_xlabel('Wafer Sequence')
ax2.set_ylabel(f'Rolling Std (window={window})')
ax2.set_title(f'Sensor Variance Drift\n{top_risk_tool} — Top 3 Sensors', fontweight='bold', color=BLUE)
ax2.legend(fontsize=8)
ax2.set_facecolor('#FAFCFF')

# 4c. Risk score gauge (heatmap style)
ax3 = fig.add_subplot(gs[1, 0])
risk_matrix = risk_df.set_index('tool')[['risk_score']].T
sns.heatmap(risk_matrix, annot=True, fmt='.0f', cmap='RdYlGn_r',
            ax=ax3, cbar=True, linewidths=2,
            vmin=0, vmax=100, annot_kws={'size': 11, 'weight': 'bold'})
ax3.set_title('Equipment Risk Heatmap\n(Red = High Risk, Green = Healthy)', 
              fontweight='bold', color=BLUE)
ax3.set_ylabel('')
ax3.set_yticklabels(['Risk\nScore'], rotation=0)

# 4d. Anomaly count vs fail rate scatter
ax4 = fig.add_subplot(gs[1, 1])
for _, row in risk_df.iterrows():
    color = risk_color(row['risk_score'])
    ax4.scatter(row['anomaly_count'], row['anomaly_fail_rate'],
                s=row['risk_score']*4, color=color, alpha=0.8, edgecolors='white', linewidth=1.5)
    ax4.annotate(row['tool'].replace('Tool-', ''), 
                 (row['anomaly_count'], row['anomaly_fail_rate']),
                 textcoords='offset points', xytext=(6, 4), fontsize=8)
ax4.axhline(y.mean()*100, color=GRAY, linestyle='--', alpha=0.7,
            label=f'Baseline fail rate: {y.mean()*100:.1f}%')
ax4.set_xlabel('Anomaly Event Count')
ax4.set_ylabel('Failure Rate During Anomalies (%)')
ax4.set_title('Anomaly Count vs Failure Rate\n(Bubble size = Risk Score)', fontweight='bold', color=BLUE)
ax4.legend(fontsize=8)

plt.savefig('../docs/fig04_equipment_risk.png', dpi=150, bbox_inches='tight')
plt.show()

# Save risk scores
risk_df.to_csv('../data/equipment_risk_scores.csv', index=False)
print('Figure 04 saved. Risk scores saved to data/equipment_risk_scores.csv')